In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .appName("TransformZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.sql.catalog.hive", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hive.catalog-impl", "org.apache.iceberg.hive.HiveCatalog") \
    .config("spark.sql.catalog.hive.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.hive.warehouse", "s3a://crypto-data-lake/transform_zone/") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .config("spark.sql.defaultCatalog", "hive") \
    .getOrCreate()

25/09/22 10:43:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
spark.sql("""
SHOW DATABASES
""").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [4]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS transform_db LOCATION 's3a://crypto-data-lake/transform_zone/'
""")

DataFrame[]

In [7]:
spark.conf.get("spark.sql.catalogImplementation")

'in-memory'

In [13]:
spark.conf.get("spark.sql.catalog.spark_catalog")